In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import pytidycensus as tc
import os
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor
from typing import Dict, Literal
from qbstyles import mpl_style
import glob
import warnings

from pyresearch import collect_acs_data

warnings.filterwarnings("ignore")

%config InlineBackend.figure_format = 'retina'

mpl_style(dark=True)

pd.set_option("future.no_silent_downcasting", True)

load_dotenv()

API_KEY: str = os.getenv("CENSUS_KEY")
DATA: str = os.getenv("DATA")

tc.set_census_api_key(API_KEY)



Census API key has been set for this session.


/tmp/ipykernel_110291/117505674.py:5: UserWarning: Mapping functions unavailable due to import error: NameError. To use mapping features, ensure all dependencies are properly installed: pip install pytidycensus[map]
  import pytidycensus as tc


## RUCC Codes

|Code|Description|Category|
|---|---|---|
|1|Counties in metro areas of 1 million population or more|Metropolitan|
|2|Counties in metro areas of 250,000 to 1 million population|Metropolitan|
|3|Counties in metro areas of fewer than 250,000 population|Metropolitan|
|4|Urban population of 20,000 or more, adjacent to a metro area|Nonmetropolitan|
|5|Urban population of 20,000 or more, not adjacent to a metro area|Nonmetropolitan|
|6|Urban population of 5,000 to 20,000, adjacent to a metro area|Nonmetropolitan|
|7|Urban population of 5,000 to 20,000, not adjacent to a metro area|Nonmetropolitan|
|8|Urban population of fewer than 5,000, adjacent to a metro area|Nonmetropolitan|
|9|Urban population of fewer than 5,000, not adjacent to a metro area|Nonmetropolitan|


In [2]:
def collect_rucc_data(data_path: str) -> pd.DataFrame:
    data = pd.read_csv(f"{data_path}/RUCC.csv")
    data.drop(columns=["Population_2020", "Description"], inplace=True)
    data.rename(columns={"RUCC_2023": "RUCC"}, inplace=True)
    data["County_Name"] = data["County_Name"].str.replace("County", "").str.strip()
    data["County_Name"] = data["County_Name"].str.replace("Parish", "").str.strip()

    data.rename(columns={"State": "state", "County_Name": "county"}, inplace=True)

    return data


rucc = collect_rucc_data(data_path=DATA)
rucc.head()


,FIPS,state,county,RUCC
0,1001,AL,Autauga,2.0
1,1003,AL,Baldwin,3.0
2,1005,AL,Barbour,6.0
3,1007,AL,Bibb,1.0
4,1009,AL,Blount,1.0


In [ ]:
# Get census data for each state
vars = {
"poverty_count": "B17001_002E",
    "white": "B01001A_001E",
    "black": "B01001B_001E",
    "total_pop": "B01003_001E",
    "am_in_ala_nat": "B01001C_001E",
    "asian": "B01001D_001E",
    "haw_pac": "B01001E_001E",
    "other": "B01001F_001E",
    "hisp_lat": "B01001I_001E",
    "male_no_hs": "C15002A_003E",
    "fem_no_hs": "C15002A_008E",
    "not_citizen": "B05001_006E",
    "median_earnings": "B08521_001E",
    "gini_index": "B19083_001E",
}



states_to_remove = [
    "PR",
    "HI",
    "GU",
    "MP",
    "AS",
    "AK",
    "VI",
]  # Filter states to continental US + D.C.

census = collect_acs_data(
    API_KEY=API_KEY,
    geography="county",
    year=2019,
    states=rucc["state"], 
    states_to_remove=states_to_remove,
    vars=vars,
    max_workers=10
    )

census["poverty"] = census["pov_under_50"] + census["pov_under_100"]
census["no_hs"] = census["male_no_hs"] + census["fem_no_hs"]

scale_to_pop = [
    "poverty",
    "no_hs",
    "white",
    "black",
    "am_in_ala_nat",
    "asian",
    "haw_pac",
    "other",
    "hisp_lat",
    "not_citizen",
]

for var in scale_to_pop:
    census[var] = round(census[var] / census["total_pop"], 2)

census.drop(
    columns=[
        "pov_under_50",
        "pov_under_100",
        "male_no_hs",
        "fem_no_hs",
        "county",
        "state",
    ],
    inplace=True,
)
census.head()


Census API key has been set for this session.


TypeError: collect_acs_data.<locals>.get_state_data() missing 1 required positional argument: 'year'

## ICE Detention Data

### Overview

From [dataset Github](https://github.com/vera-institute/ice-detention-trends/tree/main#about-the-data)

> Vera’s ICE Detention Trends dashboard reveals an unprecedented level of detail about ICE detention populations—nationally and across the 1,464 facilities in which ICE detained people—on each day of the 16-year period from fiscal year 2009 through the beginning of fiscal year 2026 (October 1, 2008, through October 15, 2025). This repository includes the aggregated data visualized in the dashboard, including information on:
>
> - Midnight population: the daily number of people detained at midnight (nationally and by facility).
> - 24-hour population: the number of people detained for any part of a given day, including those whom ICE transferred or booked-out of custody before midnight (nationally and by facility). While ICE relies solely on midnight populations in its reporting, Vera includes both types of daily populations—midnight and 24-hour—as the two can differ drastically.
> - Book-ins: the daily number of people ICE booked into custody (nationally).
> - Book-outs: the daily number of people ICE booked out of custody (nationally).
> - Facility names, locations, and types (as coded by ICE in other datasets, where available).
> 
> The original datasets included facility names and codes, but no information on location or facility type. Vera drew from additional datasets and public sources to geocode facility > locations and assign facility types. Given the lack of a comprehensive, up-to-date ICE source to assign facility types to all 1,464 facility codes in the dataset, Vera’s categorizations should be interpreted as best-known facility type. To simplify map filtering options, Vera grouped facility types assigned by ICE, as well as ones manually entered by Vera, into the following categories:
>
> - Non-Dedicated: IGSA (Inter-governmental Service Agreement).
> - Dedicated: DIGSA (Dedicated IGSA), SPC (Service Processing Center), CDF (Contract Detention Facility).
> - Federal: USMS IGA (U.S. Marshals Service Inter-governmental Agreement), BOP (Bureau of Prisons), USMS CDF (U.S. Marshals Service Contract Detention Facility), DOD (Department of Defense), MOC (Migrant Operations Center). Because ICE can be added to other federal agencies’ facility contracts or agreements through a “rider,” Vera reports federal facilities as a separate category, rather than grouped with other categories such as non-dedicated facilities.
> - Hold/Staging.
> - Family/Youth: Family, Family Staging, Juvenile. ICE’s use of the “Juvenile” facility type reflects ICE detention and does not refer to facilities used to detain unaccompanied children in the custody of the Office of Refugee Resettlement (ORR).
> - Medical: Facilities coded by ICE as “Hospital” and medical or mental health facilities manually coded by Vera.
> - Hotel: Facilities coded by ICE as “Hotel” and facilities manually coded by Vera.
> - Other/Unknown: Facilities coded by ICE as “Other” or ones for which Vera was unable to assign facility type.


### Facilities

#### Variables

> The ICE detention datasets include facility names and codes, but no information on location or facility type. Vera drew from additional datasets and public sources to geocode facility locations and assign facility types. Given the lack of a comprehensive, up-to-date ICE source to assign facility types to all facility codes in the dataset, Vera’s categorizations should be interpreted as best-known facility type.


|Variable|Type|Description|
|---|---|---|
|detention_facility_code|`string`|The unique identifier used in the ICE detention data for each facility|
|detention_facility_name|`string`|The facility name associated with the detention_facility_code in the ICE detention data|
|latitude|`numeric`|The latitude coordinate of the facility location|
|longitude|`numeric`|The longitude coordinate of the facility location|
|address|`string`|The best known facility address|
|city|`string`|The city in which the facility is located|
|county|`string`|The county in which the facility is located|
|state|`string`|The state abbreviation code. This also includes codes for U.S. territories (e.g. "PR" for "Puerto Rico") and Cuba ("CU") for facilities at Naval Station Guantanamo Bay.|
|aor|`aor`|The ICE Area of Responsibility (AOR), originally mapped by Will Craft of the Guardian US. This reflects county boundaries extracted from ICE's [field office map](https://www.ice.gov/doclib/about/offices/ero/pdf/eroFieldOffices.pdf), last updated by ICE in February 2024.|


In [16]:
if "facilities" not in locals():
    print("Processing facilities data")
    facilities = pd.read_csv(f"{DATA}/ice-detention-trends/metadata/facilities.csv")
    facilities.drop(columns=["address", "city", "zip"], inplace=True)

    facilities.rename(columns={"FIPS": "GEOID"}, inplace=True)
    facilities.drop(columns=["detention_facility_name", "state"], inplace=True)
    facilities.sort_values("county", inplace=True)
else:
    print("facilities data already processed")
facilities.head()


facilities data already processed


,detention_facility_code,county,aor,latitude,longitude,type_detailed,type_grouped
126,BOIHOLD,Ada,Salt Lake City,43.592766,-116.286069,Hold,Hold/Staging
6,ADACOID,Ada,Salt Lake City,43.606755,-116.269811,IGSA,Non-Dedicated
7,ADAIRKY,Adair,Chicago,37.103848,-85.306634,Other,Other/Unknown
114,BIINCCO,Adams,Denver,39.760890,-104.849106,Unknown,Other/Unknown
395,DENICDF,Adams,Denver,39.766325,-104.864502,CDF,Dedicated


### State Data

Facility-level population statistics for each day between October 1, 2008, and October 15, 2025, including midnight population and 24-hour population. 


|Variable|Type|Description|
|---|---|---|
|detention_facility_code|`string`|The unique identifier used in the ICE detention data for each facility|
|detention_facility_name|`string`|The facility name associated with the detention_facility_code in the ICE detention data|
|state|`string`|The state abbreviation code. This also includes codes for U.S. territories (e.g. "PR" for "Puerto Rico") and Cuba ("CU") for facilities at Naval Station Guantanamo Bay.|
|date|`date`|The day each count is reported for (`yyyy-mm-dd` format)|
|daily_pop|`numeric`|24-hour population: the number of people detained for any part of a given day, including those whom ICE transferred or booked-out of custody before midnight|
|midnight_pop|`numeric`|Midnight population: the daily number of people detained at midnight|


In [17]:
if "state_data" not in locals():
    print("Processing state detention data")
    state_dir = f"{DATA}/ice-detention-trends/facilities/by_state"
    file_pattern = os.path.join(state_dir, "*.csv")
    csvs = glob.glob(file_pattern)
    state_data = pd.concat(
        [pd.read_csv(csv) for csv in csvs], ignore_index=True, sort=False
    )
    state_data["date"] = pd.to_datetime(state_data["date"])

else:
    print("state detention data already processed")


state detention data already processed


In [18]:
print(state_data.info())
state_data.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8875424 entries, 0 to 8875423
Data columns (total 6 columns):
 #   Column                   Dtype         
---  ------                   -----         
 0   detention_facility_code  object        
 1   detention_facility_name  object        
 2   state                    object        
 3   date                     datetime64[ns]
 4   daily_pop                float64       
 5   midnight_pop             float64       
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 406.3+ MB
None


,detention_facility_code,detention_facility_name,state,date,daily_pop,midnight_pop
0,NYEPANV,Nye County Sheriff-Pahrump,NV,2008-10-01,0.0,0.0
1,NYEPANV,Nye County Sheriff-Pahrump,NV,2008-10-02,0.0,0.0
2,NYEPANV,Nye County Sheriff-Pahrump,NV,2008-10-03,0.0,0.0
3,NYEPANV,Nye County Sheriff-Pahrump,NV,2008-10-04,0.0,0.0
4,NYEPANV,Nye County Sheriff-Pahrump,NV,2008-10-05,0.0,0.0


## Incarceration Trends Dataset

See [Incarceration Trends Dataset](https://github.com/vera-institute/incarceration-trends)

> Vera’s Incarceration Trends Project (ITP) dataset provides county-level data on prison and jail incarceration and related measures over time for the entire United States.
> 
> Vera assembled this dataset primarily using information collected by the U.S. Department of Justice Bureau of Justice Statistics (BJS) and supplemented it with data collected by Vera directly from state and local agencies
when federal data was not available. The BJS data used in the ITP dataset run through 2019 for prisons and through 2023 for jails.




### Variable Descriptions for Incarceration Metrics

Due to rapid changes in the jail populations in recent years and more broadly available data on jails, Vera now produces jail population estimate (total_jail_pop) on a quarterly basis. 

Generally, this means Vera aims to measure the number of people in jail on or near these dates:
• Q1: March 31,
• Q2: June 30,
• Q3: September 30, and
• Q4: December 31.

In [64]:
cols = [
    "year",
    "quarter",
    # "state_fips",
    # "county_name",
    "fips",
    "total_incarceration",
    "total_incarceration_rate",
    "private_jail_flag",
    "regional_jail_flag",
    "jail_rated_capacity",
    "total_jail_pop",
    "female_jail_pop",
    "male_jail_pop",
    "aapi_jail_pop",
    "black_jail_pop",
    "latinx_jail_pop",
    "native_jail_pop",
    "white_jail_pop",
    "other_race_jail_pop",
    "total_jail_pretrial",
    "total_jail_from_prison",
    "total_jail_from_other_jail",
    "total_jail_from_fed",
    "total_jail_from_bia",
    "total_jail_from_bop",
    "total_jail_from_ice",
    "total_jail_from_marshals",
    "total_jail_from_other_fed",
    "total_jail_adm",
    "total_jail_dis",
    "total_jail_sentenced",

]


In [ ]:
jail_data = pd.read_csv(f'{DATA}/incarceration_trends_county.csv')
# print(list(jail_data.columns))
print(jail_data.info())
jail_data = jail_data[cols]
jail_data = jail_data.loc[(jail_data['year'] >= 2008) & (jail_data['year'] <= 2023)]
yearly_jail_data = round(jail_data.groupby(["year", "fips"]).sum())

jail_data.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245840 entries, 0 to 245839
Columns: 146 entries, year to white_male_prison_adm_rate
dtypes: float64(135), int64(3), object(8)
memory usage: 273.8+ MB
None


,year,quarter,fips,total_incarceration,total_incarceration_rate,private_jail_flag,regional_jail_flag,jail_rated_capacity,total_jail_pop,female_jail_pop,...,total_jail_from_other_jail,total_jail_from_fed,total_jail_from_bia,total_jail_from_bop,total_jail_from_ice,total_jail_from_marshals,total_jail_from_other_fed,total_jail_adm,total_jail_dis,total_jail_sentenced
1,2023,4,1001,NaN,NaN,NaN,NaN,136.0,138.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2584.0,1823.0,138.0
2,2023,3,1001,NaN,NaN,NaN,NaN,136.0,135.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2584.0,1823.0,135.0
3,2023,2,1001,NaN,NaN,NaN,NaN,136.0,148.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2584.0,1823.0,148.0
4,2023,1,1001,NaN,NaN,NaN,NaN,136.0,142.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2584.0,1823.0,142.0
5,2022,4,1001,NaN,NaN,NaN,NaN,136.0,150.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2584.0,1823.0,150.0


In [ ]:
yearly_jail_data


quarter  total_incarceration  total_incarceration_rate  \
year fips                                                            
2008 1001         2                411.0                    1163.0   
     1003         2               1526.0                    1347.0   
     1005         2                298.0                    1578.0   
     1007         2                166.0                    1069.0   
     1009         2                277.0                     744.0   
...             ...                  ...                       ...   
2023 56037        2                  0.0                       0.0   
     56039        6                  0.0                       0.0   
     56041        2                  0.0                       0.0   
     56043        2                  0.0                       0.0   
     56045        2                  0.0                       0.0   

            private_jail_flag  regional_jail_flag  jail_rated_capacity  \
year fips                                                                
2008 1001                 0.0                 0.0                131.0   
     1003                 0.0                 0.0                749.0   
     1005                 0.0                 0.0                120.0   
     1007                 0.0                 0.0                 82.0   
     1009                 0.0                 0.0                101.0   
...                       ...                 ...                  ...   
2023 56037                0.0                 0.0                  0.0   
     56039                0.0                 0.0                 45.0   
     56041                0.0                 0.0                  0.0   
     56043                0.0                 0.0                  0.0   
     56045                0.0                 0.0                  0.0   

            total_jail_pop  female_jail_pop  male_jail_pop  aapi_jail_pop  \
year fips                                                                   
2008 1001            199.0             25.0          174.0            0.0   
     1003            753.0            104.0          649.0            2.0   
     1005            121.0              6.0          115.0            0.0   
     1007             86.0              9.0           77.0            0.0   
     1009            123.0             20.0          102.0            0.0   
...                    ...              ...            ...            ...   
2023 56037             0.0              0.0            0.0            0.0   
     56039            14.0              0.0            0.0            0.0   
     56041             0.0              0.0            0.0            0.0   
     56043             0.0              0.0            0.0            0.0   
     56045             0.0              0.0            0.0            0.0   

            ...  total_jail_from_other_jail  total_jail_from_fed  \
year fips   ...                                                    
2008 1001   ...                         4.0                  1.0   
     1003   ...                        14.0                 62.0   
     1005   ...                         0.0                  2.0   
     1007   ...                         0.0                  0.0   
     1009   ...                         2.0                  1.0   
...         ...                         ...                  ...   
2023 56037  ...                         0.0                  0.0   
     56039  ...                         0.0                  0.0   
     56041  ...                         0.0                  0.0   
     56043  ...                         0.0                  0.0   
     56045  ...                         0.0                  0.0   

            total_jail_from_bia  total_jail_from_bop  total_jail_from_ice  \
year fips                                                                   
2008 1001                   0.0                  0.0                  0.0   
     1003           